In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Vitals Notebook").getOrCreate()

In [0]:
# Read the Vitals CSV file into a Spark DataFrame
bronze_path = r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/bronze/dbo.vitals.csv'
vitals_bronze_df = spark.read.csv(bronze_path, header = True, inferSchema = True)

# Vital raw rows and columns counts
print(f"Vitals file has {vitals_bronze_df.count()} rows and {len(vitals_bronze_df.columns)} columns")
vitals_bronze_df.printSchema()

display(vitals_bronze_df)

In [0]:
# -------------------------------------------
# 1. Deduplication based on "vital_id" column (Primary Key)
# -------------------------------------------

vitals_deduplicate_df = vitals_bronze_df.dropDuplicates(["vital_id"])
print(f"After deduplication: {vitals_deduplicate_df.count()} rows")
print(f"Duplicate records count is {vitals_bronze_df.count() - vitals_deduplicate_df.count()}")
display(vitals_deduplicate_df)


In [0]:
# -----------------------------
# 2. Date Standardization 
# -----------------------------
from pyspark.sql.functions import coalesce,try_to_date ,col

vitals_recordedts_df = vitals_deduplicate_df.withColumn(
    "recorded_ts",
    coalesce(
        try_to_date(col("recorded_datetime"), "yyyy-MM-dd'T'HH:mm:ss"),   # ISO 8601
        try_to_date(col("recorded_datetime"), "yyyy-MM-dd HH:mm"),         # ISO date space-sep
        try_to_date(col("recorded_datetime"), "M/d/yy H:mm"),              # US short M/D/YY
        try_to_date(col("recorded_datetime"), "MM/dd/yy HH:mm"),           # US padded
        try_to_date(col("recorded_datetime"), "dd-MM-yyyy HH:mm"),         # DD-MM-YYYY
    )
)


display(vitals_recordedts_df)

#vitals_recordedts_df.count()

In [0]:
# ----------------------------- 
# 3. Range Validation Columns
# -----------------------------

from pyspark.sql.functions import when ,lit
from pyspark.sql.types import IntegerType,DoubleType

# A. Blood Pressure - Systolic (60–250 mmHg) ----
dfA = vitals_recordedts_df.withColumn(
    "bp_systolic_valid",
    when((col("bp_systolic") > 60) & (col("bp_systolic") < 250),
         col("bp_systolic").cast(IntegerType()))
    .otherwise(lit(None))
)
vitals_bpsystolic_df = dfA.withColumn(
    "dq_bp_systolic",
    when(col("bp_systolic_valid").isNull() & col("bp_systolic").isNotNull(), 1).otherwise(0)
)
#display(vitals_bpsystolic_df)

#-------------------------------------------------------------------------------------------------------------------

# B. Blood Pressure - Diastolic (40–150 mmHg) ----
dfB = vitals_bpsystolic_df.withColumn(
    "bp_diastolic_valid",
    when((col("bp_diastolic") > 40) & (col("bp_diastolic") < 150),
         col("bp_diastolic").cast(IntegerType()))
    .otherwise(lit(None))
)
vitals_bpdiastolic_df = dfB.withColumn(
    "dq_bp_diastolic",
    when(col("bp_diastolic_valid").isNull() & col("bp_diastolic").isNotNull(), 1).otherwise(0)
)

#display(vitals_bpdiastolic_df)

#-------------------------------------------------------------------------------------------------------------------

# C. Heart Rate (20–300 BPM) ------
dfC = vitals_bpdiastolic_df.withColumn(
    "hr_valid",
    when((col("heart_rate_bpm") > 20) & (col("heart_rate_bpm") < 300),
         col("heart_rate_bpm").cast(IntegerType()))
    .otherwise(lit(None))
)
vitals_heartrate_df = dfC.withColumn(
    "dq_hr",
    when(col("hr_valid").isNull() & col("heart_rate_bpm").isNotNull(), 1).otherwise(0)
)

#display(vitals_heartrate_df)

#------------------------------------------------------------------------------------------------------------------------

# D. Respiratory Rate (4–60 breaths/min) ----
dfD = vitals_heartrate_df.withColumn(
    "rr_valid",
    when((col("respiratory_rate") > 4) & (col("respiratory_rate") < 60),
         col("respiratory_rate").cast(IntegerType()))
    .otherwise(lit(None))
)
vitals_respiratory_df = dfD.withColumn(
    "dq_rr",
    when(col("rr_valid").isNull() & col("respiratory_rate").isNotNull(), 1).otherwise(0)
)
#display(vitals_respiratory_df)

#--------------------------------------------------------------------------------------------------------------------

# E. Temperature (30.0–45.0°C) ----
dfE = vitals_respiratory_df.withColumn(
    "temp_valid",
    when((col("temperature_celsius") > 30.0) & (col("temperature_celsius") < 45.0),
         col("temperature_celsius").cast(DoubleType()))
    .otherwise(lit(None))
)
vitals_temp_df = dfE.withColumn(
    "dq_temp",
    when(col("temp_valid").isNull() & col("temperature_celsius").isNotNull(), 1).otherwise(0)
)
#display(vitals_temp_df)

#------------------------------------------------------------------------------------------------------

# F. SpO2 (50–100%) ----
dfF = vitals_temp_df.withColumn(
    "spo2_valid",
    when((col("spo2_percent") >= 50) & (col("spo2_percent") <= 100),
         col("spo2_percent").cast(DoubleType()))
    .otherwise(lit(None))
)
vitals_spO2_df = dfF.withColumn(
    "dq_spo2",
    when(col("spo2_valid").isNull() & col("spo2_percent").isNotNull(), 1).otherwise(0)
)
#display(vitals_spO2_df)

#------------------------------------------------------------------------------------------------------

# G. Weight (1–500 kg) ----
dfG = vitals_spO2_df.withColumn(
    "weight_valid",
    when((col("weight_kg") > 1) & (col("weight_kg") < 500),
         col("weight_kg").cast(DoubleType()))
    .otherwise(lit(None))
)
vitals_weight_df = dfG.withColumn(
    "dq_weight",
    when(col("weight_valid").isNull() & col("weight_kg").isNotNull(), 1).otherwise(0)
)
#display(vitals_weight_df)

#---------------------------------------------------------------------------------------------------------

# H. Height (30–250 cm) ----
dfH = vitals_weight_df.withColumn(
    "height_valid",
    when((col("height_cm") > 30) & (col("height_cm") < 250),
         col("height_cm").cast(DoubleType()))
    .otherwise(lit(None))
)
vitals_height_df = dfH.withColumn(
    "dq_height",
    when(col("height_valid").isNull() & col("height_cm").isNotNull(), 1).otherwise(0)
)
#display(vitals_height_df)

#------------------------------------------------------------------------------------------------------

# I. Pain Score (0–10, NRS scale) ----
dfI = vitals_height_df.withColumn(
    "pain_score_valid",
    when((col("pain_score") >= 0) & (col("pain_score") <= 10),
         col("pain_score").cast(IntegerType()))
    .otherwise(lit(None))
)
vitals_pain_df = dfI.withColumn(
    "dq_pain",
    when(col("pain_score_valid").isNull() & col("pain_score").isNotNull(), 1).otherwise(0)
)
#display(vitals_pain_df)

#---------------------------------------------------------------------------------------------------------------

# J. Glucose Fasting (10–1000 mg/dL) ----
dfJ = vitals_pain_df.withColumn(
    "glucose_valid",
    when((col("glucose_fasting_mg_dl") > 10) & (col("glucose_fasting_mg_dl") < 1000),
         col("glucose_fasting_mg_dl").cast(DoubleType()))
    .otherwise(lit(None))
)
vitals_glucose_df = dfJ.withColumn(
    "dq_glucose",
    when(col("glucose_valid").isNull() & col("glucose_fasting_mg_dl").isNotNull(), 1).otherwise(0)
)
#display(vitals_glucose_df)


vitals_rangestandard_df = vitals_glucose_df
display(vitals_rangestandard_df)

In [0]:
# -----------------------------
# 4. Derived Columns
# -----------------------------

from pyspark.sql.functions import round

# A. BP Category (AHA 2017 Guidelines) ----
# Normal: systolic < 120 AND diastolic < 80
# Elevated: systolic 120-129 AND diastolic < 80
# Stage 1: systolic 130-139 OR diastolic 80-89
# Stage 2: systolic >= 140 OR diastolic >= 90

vitals_bpcategory_df = vitals_rangestandard_df.withColumn(
    "bp_category",
    when(col("bp_systolic_valid").isNull() | col("bp_diastolic_valid").isNull(), lit(None))

    .when((col("bp_systolic_valid") < 120) & (col("bp_diastolic_valid") < 80), lit("Normal"))

    .when((col("bp_systolic_valid").between(120, 129)) & (col("bp_diastolic_valid") < 80), lit("Elevated"))

    .when(
        (col("bp_systolic_valid").between(130, 139)) | (col("bp_diastolic_valid").between(80, 89)),
        lit("Stage 1 Hypertension"))

    .when(
        (col("bp_systolic_valid") >= 140) | (col("bp_diastolic_valid") >= 90), 
        lit("Stage 2 Hypertension"))

    .otherwise(lit(None))
)
#display(vitals_bpcategory_df)

#-----------------------------------------------------------------------------------------------------------------

## B. Pulse Pressure ----
# Pulse Pressure = Systolic - Diastolic (cardiac risk indicator)

vitals_pulsepressure_df = vitals_bpcategory_df.withColumn(
    "pulse_pressure",
    when(col("bp_systolic_valid").isNotNull() & col("bp_diastolic_valid").isNotNull(),
         col("bp_systolic_valid") - col("bp_diastolic_valid"))
    .otherwise(lit(None))
)
#display(vitals_pulsepressure_df)

#-----------------------------------------------------------------------------------------------------------------

# C. Temperature in Fahrenheit ----
# Formula: (°C × 9/5) + 32
vitals_tempinFa_df = vitals_pulsepressure_df.withColumn(
    "temp_fahrenheit",
    when(col("temp_valid").isNotNull(),
         round((col("temp_valid") * 9 / 5) + 32, 1))
    .otherwise(lit(None))
)
#display(vitals_tempinFa_df)

#------------------------------------------------------------------------------------------------

# D. SpO2 Category ----
vitals_spO2category_df = vitals_tempinFa_df.withColumn(
    "spo2_category",
    when(col("spo2_valid").isNull(), lit("Unknown"))
    .when(col("spo2_valid") >= 95, lit("Normal"))
    .when(col("spo2_valid").between(90, 94), lit("Mild Hypoxia"))
    .when(col("spo2_valid").between(85, 89), lit("Moderate Hypoxia"))
    .when(col("spo2_valid") < 85, lit("Severe Hypoxia"))
    .otherwise(lit("Unknown"))
)
#display(vitals_spO2category_df)

#----------------------------------------------------------------------------------------------------

# E. BMI Recalculated ----
# BMI = weight(kg) / (height(m))^2
vitals_bmicalculation_df = vitals_spO2category_df.withColumn(
    "bmi_calculated",
    when(col("weight_valid").isNotNull() & col("height_valid").isNotNull(),
         round(col("weight_valid") / ((col("height_valid") / 100) ** 2), 1))
    .otherwise(lit(None))
)
#display(vitals_bmi_df)

#--------------------------------------------------------------------------------------------------------

# F. BMI Category (WHO Classification) ----
vitals_bmicategory_df = vitals_bmicalculation_df.withColumn(
    "bmi_category",
    when(col("bmi_calculated").isNull(), lit("Unknown"))
    .when(col("bmi_calculated") < 18.5, lit("Underweight"))
    .when(col("bmi_calculated").between(18.5, 24.9), lit("Normal"))
    .when(col("bmi_calculated").between(25.0, 29.9), lit("Overweight"))
    .when(col("bmi_calculated") >= 30, lit("Obese"))
    .otherwise(lit("Unknown"))
)
#display(vitals_bmicategory_df)


vitals_derived_df = vitals_bmicategory_df
display(vitals_derived_df)

In [0]:
#select the columns for Final silver layer 

vitals_silver_df = vitals_derived_df.select(
    # --- Identifiers ---
    "vital_id",
    "encounter_id",
    "patient_id",

    # --- Timestamps ---
    "recorded_ts",
    "ingestion_timestamp",

    # --- Who recorded ---
    "recorded_by",

    # --- Blood Pressure ---
    "bp_systolic_valid",
    "bp_diastolic_valid",
    "bp_category",
    "pulse_pressure",

    # --- Heart & Respiration ---
    "hr_valid",
    "rr_valid",

    # --- Temperature ---
    "temp_valid",
    "temp_fahrenheit",

    # --- SpO2 ---
    "spo2_valid",
    "spo2_category",

    # --- Weight / Height / BMI ---
    "weight_valid",
    "height_valid",
    "bmi_calculated",
    "bmi_category",

    # --- Pain & Glucose ---
    "pain_score_valid",
    "glucose_valid",

    # --- DQ ---
    "dq_bp_systolic",
    "dq_bp_diastolic",
    "dq_hr",
    "dq_rr",
    "dq_temp",
    "dq_spo2",
    "dq_weight",
    "dq_height",
    "dq_pain",
    "dq_glucose",
)

display(vitals_silver_df)


In [0]:
# ----------------------------------------
#  Write to Silver layer in delta format
# ----------------------------------------


TABLE_NAME = "vitals"  

silver_table_path = f"abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/{TABLE_NAME}"

# ── OVERWRITE (Full Load) ────────────────────────────────────
vitals_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print(f"✅ Data written to Silver layer: {silver_table_path}")